# GETM Dutch Wadden Sea — wave validation against RWS observations, 2015

Compares `dws_500m.3d.2015??.nc` from one model run with the Rijkswaterstaat
wave buoys and tide gauges in `OBS_ROOT/processed/`.

| Model | Observation | Note |
|---|---|---|
| `Hs_out` | `hm0` | significant wave height, direct |
| `Tz_out` | `tp` | peak period (named Tz in the model output, computed as Tp) |
| `elev` | `wl` | water level — **aliased, see below** |
| `EUWIND`/`EVWIND` | KNMI wind | forcing sanity check, not model skill |

The model variable is called `Tz_out`, but the GETM–ERSEM–BFM wave scheme
computes the **peak period**, so it is compared with the observed peak period
`tp` and labelled Tp in every figure. Observed Tp is masked with `valid_tp`
(peak period saturates at the edges of the 0.03–0.50 Hz band when the sea is
flat), so it has fewer valid days than Hs.

## What it cannot validate, and why

**Currents.** There are no current-meter observations in the Dutch Wadden Sea in
2015 (RWS current data start at Eemshaven in 2020). The `u`/`v` code (§11) is
ready but finds nothing until a year ≥ 2020 is run.

**Tidal dynamics from daily snapshots.** With one instantaneous value per day at
00:00, the M2 tide (12.42 h) aliases to
$\left|f_{M2} - 2f_s\right|^{-1} = |1.9323 - 2|^{-1} \approx 14.8$ days, so daily
`elev` shows a spurious ~15-day oscillation and even a perfect tidal model would
score badly against instantaneous observations. Water level is reported (§10)
with that caveat and must not be read as tidal skill. Waves are far less
affected: `Hs` is driven by wind on synoptic time scales, so a daily sample is a
coarse but honest sample of the wave climate.

**Statistical independence.** 365 daily samples of a quantity with a 1–2 day
decorrelation time give roughly 100–200 effective degrees of freedom, not 365.

## Three model-file details

1. Each monthly file carries 32 time steps (00:00 on the 1st through 00:00 on
   the 1st of the next month), so concatenating files **duplicates every month
   boundary**. §5 collapses them.
2. `Hs_out` and `Tz_out` are **`-9998` over the whole domain on the first frame
   of every file**. That is not the declared `_FillValue` (`-9999`), so xarray
   does not mask it; §5 does.
3. Together with the `averaged = 1, 0` attribute those variables carry, the
   empty first frame says the wave fields are **means over the interval ending
   at the time stamp**. §6 tests that against the observations.

## Figures

All figures go to `OUT_DIR/figures/` as vector PDF + 600 dpi PNG + a CSV of the
plotted numbers, in the shared house style (`figstyle.py`).

| File | Content |
|---|---|
| `fig01_wave_stations` | model domain, bathymetry and compared stations |
| `fig02_wave_timeseries_key` | Hs and Tp at the four best-performing stations |
| `fig03_wave_scatter` | pooled model vs observed Hs and Tp |
| `fig04_wave_taylor` | Taylor diagram for Hs, coloured by bias |
| `fig05_wave_bias_map` | station bias of Hs and Tp on the map |
| `fig06_wave_bias_by_sea_state` | Hs bias and RMSE by observed sea state |
| `fig07_wave_reachable_bias` | could another cell nearby have removed the Hs bias? |
| `fig08_wave_cell_cross_check` | does the Hs-optimal cell also help Tp? |
| `figS1`–`figS4` | Hs and Tp time series and scatter at every station |
| `figS5_wave_waterlevel` | water level at 00:00 and its aliased difference |

---
## 1. Configuration

The only cell you should normally need to edit.

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle
from scipy.spatial import cKDTree

sys.path.insert(0, str(Path.cwd()))          # figstyle.py sits next to this notebook
import figstyle as fs

warnings.filterwarnings("ignore", category=RuntimeWarning)
fs.use_style()

# --- paths -------------------------------------------------------------------
OBS_ROOT   = Path("/export/lv9/projects/dws/results/validation/waves/")
MODEL_DIR  = Path("/export/lv9/projects/dws/model_output/archived_runs/effective_fetch/spinup_02/")
MODEL_GLOB = "dws_500m.3d.2015??.nc"
RUN_NAME   = MODEL_DIR.name                           # e.g. spinup_02
OUT_DIR    = OBS_ROOT / "output" / MODEL_DIR.parent.name / RUN_NAME
FIG_DIR    = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

YEAR = 2015

# --- matching ----------------------------------------------------------------
MAX_MATCH_KM = 3.0        # reject a station further than this from a wet cell
# A (2R+1)x(2R+1) window is extracted around each station so that alternative
# cell choices can be compared without re-reading the files: R=2 is 2.5 km.
NEIGHBOURHOOD_R = 2
CELL_STRATEGY = "nearest"  # "nearest" | "depth-matched" | "window-median" (see §9)
MIN_MODEL_DEPTH_M = 0.5
MIN_PAIRS = 30             # skip station/variable pairs with fewer matched days
MATCH_TOL = pd.Timedelta("10min")   # obs <-> model instantaneous tolerance

# --- observation conditioning ------------------------------------------------
# The RWS tables carry qc_* flags and validity masks (README section 4b). With
# this on, values the instrument could not have measured are blanked before
# anything is computed from them. False compares against the raw values.
USE_QC_MASKS = True

# GETM writes -9998 across the domain into the time-averaged fields on the first
# frame of every file; anything below this is treated as missing.
SENTINEL_BELOW = -9990.0

# model variable -> observation column
PAIRS = {
    "Hs_out": "hm0",
    "Tz_out": "tp",     # named Tz in the model output, but computed as the peak period
    "elev":   "wl",
}
# Orbital velocity is deliberately absent: it is derived on both sides, so a
# disagreement confounds wave physics with two definitions and a bed level.
MODEL_VARS  = ["elev", "u", "v", "Hs_out", "Tz_out", "TauW", "TauC", "EUWIND", "EVWIND", "dry_z"]
STATIC_VARS = ["lonc", "latc", "bathymetry", "convc"]

# --- figures -----------------------------------------------------------------
UNITS = {"Hs_out": "m", "Tz_out": "s", "elev": "m"}
LABEL = {"Hs_out": "Hs", "Tz_out": "Tp", "elev": "Water level"}
OBS_LABEL = {"hm0": "Hm0", "tm02": "Tm02", "tp": "Tp", "wl": "water level"}
# Stations for Fig. 2: the top four of the ranking printed with Fig. 2 (spinup_02,
# 2015), listed west to east. Set to None to take the top four of a new run.
KEY_STATIONS = ["eierlandsegat", "stortemelk.oost", "schiermonnikoog.noord", "westereems.west"]
KEY_MIN_DAYS = 200         # paired days needed for both Hs and Tp to be a key station
TAYLOR_VARS = ["Hs_out"]   # variables with a Taylor diagram (Fig. 4)
PER_PAGE = 8               # stations per supplementary time-series page
MAP_EXTENT  = (4.0, 6.9, 52.8, 53.78)       # lon0, lon1, lat0, lat1
ZOOM_EXTENT = (5.42, 5.84, 53.29, 53.535)   # Amelander Zeegat buoy array
ZOOM_PREFIX = "amelanderzeegat."            # stations shown in the zoom panel
MODEL_COLOUR = fs.run_colours([RUN_NAME])[RUN_NAME]
RAW_COLOUR = "#c9c7c0"                      # 10-min observations, background layer

print("obs root :", OBS_ROOT)
print("model dir:", MODEL_DIR, "->", "FOUND" if MODEL_DIR.is_dir() else "NOT FOUND (edit MODEL_DIR)")
print("figures  :", FIG_DIR)

---
## 2. Observations, conditioned

Reads `processed/csv/<station>.csv.gz` (10-minute, UTC, SI units) and keeps the
stations with data in the target year.

The tables carry six `qc_*` flags and three validity masks (README section 4b).
Parts of this archive are not measurements: peak period saturates at both edges
of the 0.03–0.50 Hz band when the sea is flat, `Hm0` is quantised at 0.01 m so
anything under ~0.05 m is arithmetic, and intertidal step gauges keep reporting
after the flat has dried. Masking here, at load time, means every score and
figure below uses the same conditioned data.

In [ ]:
MASK_FOR = {"hm0": "valid_hm0", "h13": "valid_hm0", "hmax": "valid_hm0",
            "tm02": "valid_tm02", "tm_10": "valid_tm02", "tm02_hf": "valid_tm02",
            "tm_10_hf": "valid_tm02", "tp": "valid_tp", "fp": "valid_tp"}
EMERGENT_BLANKS = ["wl", "depth", "u_orb", "u_orb_rms"]
qc_removed = []


def apply_qc(d, code_):
    """Blank observations the flags say are not measurements (values -> NaN)."""
    before = {c: int(d[c].notna().sum()) for c in d.columns
              if c in MASK_FOR or c in EMERGENT_BLANKS}
    for col, mask in MASK_FOR.items():
        if col in d.columns and mask in d.columns:
            d.loc[d[mask] != 1, col] = np.nan
    if "qc_emergent" in d.columns:
        cols = [c for c in EMERGENT_BLANKS if c in d.columns]
        if cols:
            d.loc[d["qc_emergent"] == 1, cols] = np.nan
    for c, n0 in before.items():
        n1 = int(d[c].notna().sum())
        if n0 and n1 < n0:
            qc_removed.append({"station": code_, "column": c, "kept": n1,
                               "was": n0, "removed_pct": 100 * (1 - n1 / n0)})
    return d


ov = pd.read_csv(OBS_ROOT / "processed" / "station_overview.csv")

OBS = {}
for code_ in ov["code"]:
    fp = OBS_ROOT / "processed" / "csv" / f"{code_}.csv.gz"
    if not fp.exists():
        continue
    d = pd.read_csv(fp, index_col=0, parse_dates=True).loc[f"{YEAR}-01-01":f"{YEAR}-12-31"]
    if d.empty or not d.notna().any().any():
        continue
    OBS[code_] = apply_qc(d, code_) if USE_QC_MASKS else d

meta = ov.set_index("code").loc[list(OBS)]
counts = pd.DataFrame({c: {v: int(OBS[c][v].notna().sum()) if v in OBS[c] else 0
                           for v in PAIRS.values()} for c in OBS}).T
counts["lat"], counts["lon"] = meta["lat"], meta["lon"]
print(f"{len(OBS)} stations with {YEAR} data\n")
display(counts.sort_values("hm0", ascending=False))

if USE_QC_MASKS:
    qc_tab = pd.DataFrame(qc_removed)
    if qc_tab.empty:
        print("\nQC masks on: nothing needed blanking in this year.")
    else:
        piv = (qc_tab.pivot_table(index="station", columns="column",
                                  values="removed_pct", aggfunc="first").round(1).fillna(0.0))
        print("\nQC masks on - per cent of each column blanked as not-a-measurement:")
        display(piv.loc[piv.max(axis=1).sort_values(ascending=False).index])
else:
    print("\nQC masks OFF - comparing against the raw delivered values.")


def short_name(code_):
    """Compact station label for figures: 'AZG 1.1', 'Stortemelk E', ..."""
    if code_.startswith(ZOOM_PREFIX):
        return "AZG " + code_[len(ZOOM_PREFIX):]
    name = str(ov.set_index("code").loc[code_, "name"]) if code_ in set(ov["code"]) else code_
    parts = [p.strip() for p in name.split(",")]
    abbrev = {"noord": "N", "zuid": "S", "oost": "E", "west": "W", "boei": ""}
    tail = [abbrev.get(p.lower(), p) for p in parts[1:]]
    tail = [t for t in tail if t]
    if tail and all(len(t) <= 2 for t in tail):
        return f"{parts[0]} {''.join(tail)}"
    return " ".join([parts[0]] + tail)

---
## 3. Model grid

`bathymetry` is positive-down depth below the model reference level; land is
masked by its `_FillValue`. `lonc`/`latc` carry `_FillValue = -999` outside the
computational domain, which breaks `pcolormesh` and would make a KD-tree match
stations to nonsense. The holes are filled (smooth trend in index space plus
interpolated residual; `fs.fill_coord`) and the wet mask also requires finite
original coordinates.

In [ ]:
files = sorted(MODEL_DIR.glob(MODEL_GLOB))
assert files, f"no model files matching {MODEL_GLOB} in {MODEL_DIR}"
print(f"{len(files)} model files:", ", ".join(f.name for f in files[:3]), "...")

with xr.open_dataset(files[0]) as d0:
    lon_raw = np.asarray(np.ma.filled(d0["lonc"].values, np.nan), float)
    lat_raw = np.asarray(np.ma.filled(d0["latc"].values, np.nan), float)
    lon2d = fs.fill_coord(lon_raw, "lonc")
    lat2d = fs.fill_coord(lat_raw, "latc")
    bathy = np.asarray(np.ma.filled(d0["bathymetry"].values, np.nan), float)
    convc = d0["convc"].values if "convc" in d0 else None
    print("grid", lon2d.shape, "| time units:", d0["time"].encoding.get("units", "?"))
    missing = [v for v in MODEL_VARS if v not in d0.variables]
    if missing:
        print("NOT in file (will be skipped):", missing)
    for v in ("Hs_out", "elev"):
        if v in d0.variables:
            print(f"  {v}: attrs = { {k: val for k, val in d0[v].attrs.items() if k in ('averaged', 'long_name')} }")

coord_ok = np.isfinite(lon_raw) & np.isfinite(lat_raw)
wet = np.isfinite(bathy) & (bathy > MIN_MODEL_DEPTH_M) & coord_ok
assert wet.any(), "no usable wet cells - check bathymetry and MIN_MODEL_DEPTH_M"
print(f"\nwet cells: {wet.sum():,} of {wet.size:,}")
if convc is not None:
    print(f"grid rotation convc: {np.nanmin(convc):.2f}..{np.nanmax(convc):.2f} deg "
          "(u/v are labelled 'global x/y direction', i.e. already geographic)")

---
## 4. Station → grid matching

Nearest **wet** cell on a local equirectangular projection. Stations further
than `MAX_MATCH_KM` from any wet cell are outside the domain (or on a flat the
model treats as land) and are dropped. A tidal-flat station may legitimately
sit in a cell that dries; that is a model–reality difference worth seeing, not a
matching error, so drying is not a rejection criterion.

In [ ]:
R_EARTH = 6371.0
lat0 = float(np.nanmean(lat2d[wet]))


def _xy(lon, lat):
    return (np.radians(lon) * np.cos(np.radians(lat0)) * R_EARTH, np.radians(lat) * R_EARTH)


jw, iw = np.nonzero(wet)
tree = cKDTree(np.column_stack(_xy(lon2d[wet], lat2d[wet])))

rows = []
for code_ in OBS:
    slat, slon = float(meta.loc[code_, "lat"]), float(meta.loc[code_, "lon"])
    dist, k = tree.query(_xy(slon, slat))
    j, i = int(jw[k]), int(iw[k])
    rows.append({"code": code_, "label": short_name(code_), "lat": slat, "lon": slon,
                 "j": j, "i": i, "dist_km": float(dist),
                 "model_lat": float(lat2d[j, i]), "model_lon": float(lon2d[j, i]),
                 "model_depth_m": float(bathy[j, i])})
match = pd.DataFrame(rows).set_index("code").sort_values("dist_km")
match["inside"] = match["dist_km"] <= MAX_MATCH_KM
print(f"{int(match['inside'].sum())} of {len(match)} stations inside the domain "
      f"(<= {MAX_MATCH_KM} km from a wet cell)\n")
display(match[["label", "lat", "lon", "dist_km", "model_depth_m", "inside"]])
dropped = match.index[~match["inside"]].tolist()
if dropped:
    print("\ndropped (outside domain):", ", ".join(dropped))
STN = match.index[match["inside"]].tolist()

# --- the neighbourhood stencil around each matched cell ------------------------
R = NEIGHBOURHOOD_R
OFFSETS = [(dj, di) for dj in range(-R, R + 1) for di in range(-R, R + 1)]
CENTRE = OFFSETS.index((0, 0))
ny, nx = lon2d.shape
_J = match.loc[STN, "j"].to_numpy()[:, None] + np.array([o[0] for o in OFFSETS])[None, :]
_I = match.loc[STN, "i"].to_numpy()[:, None] + np.array([o[1] for o in OFFSETS])[None, :]
_in = (_J >= 0) & (_J < ny) & (_I >= 0) & (_I < nx)
WIN_J, WIN_I = np.clip(_J, 0, ny - 1), np.clip(_I, 0, nx - 1)
WIN_OK = _in & wet[WIN_J, WIN_I]
WIN_DEPTH = np.where(WIN_OK, bathy[WIN_J, WIN_I], np.nan)
_spread = np.nanmax(WIN_DEPTH, axis=1) - np.nanmin(WIN_DEPTH, axis=1)
print(f"\nneighbourhood: {2 * R + 1}x{2 * R + 1} cells (~{(2 * R + 1) * 0.5:.1f} km across); "
      f"usable cells per station: min {WIN_OK.sum(axis=1).min()}, max {WIN_OK.sum(axis=1).max()}")
print(f"model depth range within the window: median {np.nanmedian(_spread):.1f} m, "
      f"max {np.nanmax(_spread):.1f} m")

### Map helpers and Fig. 1 — domain and stations

Land is GSHHG full resolution (vector), water is shaded by model depth, and the
model's computational domain is outlined in the run colour. The Amelander
Zeegat array (12 buoys, in pairs about 200 m apart) is shown in a zoom panel;
co-located markers are nudged apart by a few points and tied to their true
position by a hairline.

In [ ]:
LAND = fs.load_land(MAP_EXTENT)
LAT0_MAP = 0.5 * (MAP_EXTENT[2] + MAP_EXTENT[3])


def aspect(ext):
    return (ext[1] - ext[0]) * np.cos(np.radians(LAT0_MAP)) / (ext[3] - ext[2])


def wave_map(ax, extent, zoom=False, labels=(True, True)):
    fs.setup_map(ax, extent, lat0=LAT0_MAP, xstep=0.1 if zoom else 0.5,
                 ystep=0.1 if zoom else 0.2, labels=labels)
    fs.add_bathymetry(ax, lon2d, lat2d, np.where(np.isfinite(bathy) & coord_ok, bathy, np.nan))
    fs.add_land(ax, LAND)
    fs.add_domain_outline(ax, lon_raw, lat_raw, MODEL_COLOUR, valid=coord_ok)
    if not zoom:
        fs.add_water_label(ax, 4.55, 53.55, "North Sea")
        fs.add_water_label(ax, 5.22, 53.13, "Wadden Sea")
        z = ZOOM_EXTENT
        ax.add_patch(Rectangle((z[0], z[2]), z[1] - z[0], z[3] - z[2], fill=False,
                               ec=fs.INK, lw=0.6, zorder=9))
    fs.add_scalebar(ax, 5 if zoom else 20, "lower right")


def map_figure(nrows, extra_mm=14, key_lines=0):
    """(full domain | zoom) map pairs of equal height per row, plus a key row."""
    wa, wb = aspect(MAP_EXTENT), aspect(ZOOM_EXTENT)
    w_maps = 183 - 24                       # leave room for tick labels / colourbar
    h_row = w_maps / (wa + wb)
    key_h = 2.8 * key_lines + 2 if key_lines else 0
    fig = plt.figure(figsize=(183 * fs.MM, (nrows * h_row + key_h + extra_mm) * fs.MM))
    gs = fig.add_gridspec(nrows + (1 if key_lines else 0), 2, width_ratios=[wa, wb],
                          height_ratios=[h_row] * nrows + ([key_h] if key_lines else []))
    axes = np.array([[fig.add_subplot(gs[r, c]) for c in range(2)] for r in range(nrows)])
    for ax in axes[:, 1]:
        ax.yaxis.tick_right()
        ax.tick_params(axis="y", which="both", labelright=True, labelleft=False)
    key_ax = fig.add_subplot(gs[nrows, :]) if key_lines else None
    return fig, axes, key_ax


def zoom_letter(ax, letter):
    z = ZOOM_EXTENT
    ax.annotate(letter, (z[0], z[3]), xytext=(1.5, 1.5), textcoords="offset points",
                ha="left", va="bottom", fontsize=7, fontweight="bold", zorder=9)


def plot_station_markers(ax, lons, lats, repel=False, **kw):
    """Markers at station positions; optionally nudged apart with hairline leaders."""
    lons, lats = np.asarray(lons, float), np.asarray(lats, float)
    x, y = (fs.repel_points(ax, lons, lats, min_sep_pt=kw.get("s", 16) ** 0.5 + 0.8)
            if repel and len(lons) > 1 else (lons, lats))
    for x0, y0, x1, y1 in zip(lons, lats, x, y):
        if np.hypot(x1 - x0, y1 - y0) > 1e-6:
            ax.plot([x0, x1], [y0, y1], color=fs.INK3, lw=0.3, zorder=7)
    return ax.scatter(x, y, zorder=8, **kw), x, y


# Station numbers, used on every map and in the Taylor diagram: west to east,
# with the Amelander Zeegat array kept together in its own order.
def station_numbers(stations):
    lon = match.loc[stations, "lon"]
    azg = sorted([c_ for c_ in stations if c_.startswith(ZOOM_PREFIX)],
                 key=lambda c_: [int(v) for v in c_[len(ZOOM_PREFIX):].split(".")])
    blocks = [(lon[c_], [c_]) for c_ in stations if c_ not in azg]
    if azg:
        blocks.append((lon[azg].mean(), azg))
    order = [c_ for _, grp in sorted(blocks, key=lambda b: b[0]) for c_ in grp]
    return {c_: k for k, c_ in enumerate(order, 1)}


NUMBER = station_numbers(STN)


def station_key(ax, stations, ncols=4, note=None):
    """Number -> station name, in columns (numbers right-aligned)."""
    ax.axis("off")
    # numbers end 4 mm and names start 5.5 mm into each key column
    w_mm = ax.get_position().width * ax.figure.get_figwidth() * 25.4
    num_x, name_x = 4.0 / w_mm, 5.5 / w_mm
    items = sorted(stations, key=NUMBER.get)
    per = int(np.ceil(len(items) / ncols))
    for k in range(ncols):
        chunk = items[k * per:(k + 1) * per]
        if not chunk:
            continue
        x0 = k / ncols
        opts = dict(transform=ax.transAxes, va="top", fontsize=6, linespacing=1.3)
        ax.text(x0 + num_x, 1.0, "\n".join(str(NUMBER[c_]) for c_ in chunk), ha="right",
                fontweight="bold", **opts)
        ax.text(x0 + name_x, 1.0, "\n".join(short_name(c_) for c_ in chunk), ha="left",
                **opts)
    if note:
        ax.text(0.0, 0.0, note, transform=ax.transAxes, ha="left", va="bottom", fontsize=6,
                color=fs.INK3)


ins = match[match["inside"]]
out = match[~match["inside"]]
in_zoom = ins.index.str.startswith(ZOOM_PREFIX)
main, azg = ins[~in_zoom], ins[in_zoom]
n_lines = int(np.ceil(len(ins) / 4)) + (1 if len(out) else 0)

fig, axes, key_ax = map_figure(1, key_lines=n_lines)
ax_a, ax_b = axes[0]
wave_map(ax_a, MAP_EXTENT)
wave_map(ax_b, ZOOM_EXTENT, zoom=True, labels=(True, False))
zoom_letter(ax_a, "b")
ax_a.scatter(main["lon"], main["lat"], s=12, facecolor="white", edgecolor=fs.INK,
             linewidth=0.7, zorder=8)
ax_a.scatter(azg["lon"], azg["lat"], s=3, color=fs.INK, zorder=8)
ax_a.scatter(out["lon"], out["lat"], s=10, marker="x", color=fs.INK3, linewidth=0.7, zorder=8)
_, zx, zy = plot_station_markers(ax_b, azg["lon"], azg["lat"], repel=True, s=14,
                                 facecolor="white", edgecolor=fs.INK, linewidth=0.7)
station_key(key_ax, list(ins.index),
            note=("\u00d7 outside the model domain: " + ", ".join(out["label"])) if len(out) else None)
handles = [Line2D([], [], ls="", marker="o", ms=3.4, mfc="white", mec=fs.INK, mew=0.7,
                  label="compared station"),
           Line2D([], [], ls="", marker="x", ms=3.2, color=fs.INK3, mew=0.7,
                  label=f"outside model domain (> {MAX_MATCH_KM:g} km)"),
           Line2D([], [], color=MODEL_COLOUR, lw=0.7, label=f"model domain ({RUN_NAME})")]
fig.legend(handles=handles, loc="outside upper center", ncol=3)
fs.panel_label(ax_a, "a"); fs.panel_label(ax_b, "b")
fs.place_labels(ax_a, main["lon"], main["lat"], [str(NUMBER[c_]) for c_ in main.index],
                fontsize=6, avoid_x=np.r_[azg["lon"], out["lon"]],
                avoid_y=np.r_[azg["lat"], out["lat"]])
fs.place_labels(ax_b, zx, zy, [str(NUMBER[c_]) for c_ in azg.index], fontsize=6)
fs.save_figure(fig, "fig01_wave_stations", FIG_DIR,
               data=match.assign(number=[NUMBER.get(c_) for c_ in match.index])[
                   ["number", "label", "lat", "lon", "inside", "dist_km", "model_lat",
                    "model_lon", "model_depth_m"]])

---
## 5. Extract model time series at the stations

One pass per monthly file, indexing only the stencil cells (no `dask` needed).
The `-9998` first frame is masked, and at duplicated month boundaries the copy
with more valid data is kept — in the later file that frame is the averaged
fields' empty first frame. The result is cached as
`model_at_stations_<year>.nc`; the cache is reused only if it matches the
current stencil and station set.

In [ ]:
CACHE = OUT_DIR / f"model_at_stations_{YEAR}.nc"


def mask_sentinels(ds):
    """-9998 / -9999 / -99999 are all 'no data' here; only -9999 is declared."""
    for v in ds.data_vars:
        if ds[v].dtype.kind == "f":
            ds[v] = ds[v].where(ds[v] > SENTINEL_BELOW)
    return ds


def extract(files, jj, ii, station_names):
    """Pull the whole (station, win) stencil in one pass over the files."""
    ok = xr.DataArray(WIN_OK, dims=("station", "win"))
    frames = []
    for f in files:
        with xr.open_dataset(f) as d:
            keep = [v for v in MODEL_VARS + STATIC_VARS if v in d.variables]
            pt = d[keep].isel(yc=xr.DataArray(jj, dims=("station", "win")),
                              xc=xr.DataArray(ii, dims=("station", "win"))).load()
        pt = mask_sentinels(pt).where(ok)
        dead = {v: int(pt[v].isnull().all(dim=["station", "win"]).sum())
                for v in pt.data_vars if "time" in pt[v].dims}
        dead = {k: n for k, n in dead.items() if n}
        print(f"  {f.name}: {pt.sizes.get('time', 0)} steps"
              + (f"  | all-missing frames: {dead}" if dead else ""), flush=True)
        frames.append(pt)
    out = xr.concat(frames, dim="time").assign_coords(station=("station", station_names))
    for v in STATIC_VARS:
        if v in out and "time" in out[v].dims:
            out[v] = out[v].isel(time=0, drop=True)
    score = np.zeros(out.sizes["time"])
    for v in out.data_vars:
        if "time" in out[v].dims:
            score += out[v].notnull().sum(dim=[d for d in out[v].dims if d != "time"]).values
    t = out["time"].values
    order = np.lexsort((-score, t))
    ts = t[order]
    first = np.r_[True, ts[1:] != ts[:-1]]
    out = out.isel(time=order[first]).sortby("time")
    print(f"\n{len(t)} steps read, {len(t) - int(first.sum())} duplicate time stamps "
          f"collapsed (keeping the more complete copy), {out.sizes['time']} kept")
    return out


MODW = None
if CACHE.exists():
    cached = xr.open_dataset(CACHE)
    if (cached.sizes.get("win") == len(OFFSETS) and cached.sizes.get("station") == len(STN)
            and list(map(str, cached["station"].values)) == list(STN)):
        MODW = cached
        print("loaded cached extraction:", CACHE.name)
    else:
        cached.close()
        print(f"cache {CACHE.name} does not match the current stencil - re-extracting")
if MODW is None:
    MODW = extract(files, WIN_J, WIN_I, STN)
    MODW.to_netcdf(CACHE)
    print("cached ->", CACHE)


def select_cell(modw, how, target_depth=None):
    """Reduce the (station, win) stencil to one series per station."""
    if how == "nearest":
        return modw.isel(win=CENTRE)
    if how == "window-median":
        return modw.median(dim="win", skipna=True, keep_attrs=True)
    if how == "depth-matched":
        if target_depth is None:
            raise ValueError("depth-matched needs a target depth per station")
        diff = np.abs(WIN_DEPTH - np.asarray(target_depth, float)[:, None])
        diff = np.where(np.isfinite(diff), diff, np.inf)
        k = np.where(np.isinf(diff.min(axis=1)), CENTRE, np.argmin(diff, axis=1))
        return modw.isel(win=xr.DataArray(k, dims="station"))
    raise ValueError(f"unknown strategy {how!r}")


MOD = select_cell(MODW, CELL_STRATEGY)
mtimes = pd.DatetimeIndex(MOD["time"].values)
print(f"\ncell strategy: {CELL_STRATEGY!r}  (compared in section 9)")
print(f"model time: {mtimes[0]} .. {mtimes[-1]}  (n={len(mtimes)})")

In [ ]:
# Which variables carry data, and where are the holes?
cov = {}
for v in MOD.data_vars:
    if "time" not in MOD[v].dims:
        continue
    ok = MOD[v].notnull().any(dim="station").values
    cov[v] = {"frames_with_data": int(ok.sum()), "frames_empty": int((~ok).sum()),
              "first_valid": str(mtimes[ok][0])[:16] if ok.any() else "-",
              "last_valid": str(mtimes[ok][-1])[:16] if ok.any() else "-"}
cov = pd.DataFrame(cov).T
display(cov)
waves = [v for v in ("Hs_out", "Tz_out") if v in cov.index]
if waves and "elev" in cov.index:
    lag = pd.Timestamp(cov.loc[waves[0], "first_valid"]) - pd.Timestamp(cov.loc["elev", "first_valid"])
    print(f"\nWave fields start {lag} after elev"
          + (" - consistent with a field averaged over the preceding interval."
             if lag >= pd.Timedelta("1D") else "."))

---
## 6. Is the wave output instantaneous or a daily average?

An instantaneous 00:00 value must be compared with the observation *at* 00:00; a
daily mean with a daily mean of the observations. Getting it wrong inflates the
scatter. Two pieces of evidence already point to "mean of the previous 24 h":
the `averaged = 1, 0` attribute on `Hs_out`, `Tz_out`, `TauW`, `TauC` (and not
on `elev`, `u`, `v`), and the empty first frame of every file. The four
conventions are still fitted rather than assumed; a winner must beat that prior
by more than 5 % of RMSE to overrule it.

In [ ]:
def window_mean(series, times, lo_h, hi_h, min_frac=0.5):
    """Mean of `series` over (t+lo_h, t+hi_h] for each t, else NaN."""
    s = series.dropna()
    if s.empty:
        return pd.Series(np.nan, index=times)
    need = min_frac * (hi_h - lo_h) * 6          # 6 samples per hour
    vals = []
    for t in times:
        w = s.loc[t + pd.Timedelta(hours=lo_h): t + pd.Timedelta(hours=hi_h)]
        vals.append(w.mean() if len(w) >= need else np.nan)
    return pd.Series(vals, index=times)


def obs_at(series, times, how):
    if how == "instantaneous":
        s_ = series.dropna()
        s_ = s_[~s_.index.duplicated(keep="first")].sort_index()
        return s_.reindex(times, method="nearest", tolerance=MATCH_TOL)
    if how == "mean of previous 24 h":
        return window_mean(series, times, -24, 0)
    if how == "mean of following 24 h":
        return window_mean(series, times, 0, 24)
    if how == "centred 24 h mean":
        return window_mean(series, times, -12, 12)
    raise ValueError(how)


CONVENTIONS = ["instantaneous", "mean of previous 24 h", "mean of following 24 h",
               "centred 24 h mean"]
res = []
for how in CONVENTIONS:
    o_all, m_all = [], []
    for c_ in STN:
        if "hm0" not in OBS[c_]:
            continue
        o = obs_at(OBS[c_]["hm0"], mtimes, how)
        m = pd.Series(MOD["Hs_out"].sel(station=c_).values, index=mtimes)
        ok = o.notna() & m.notna()
        if ok.sum() >= MIN_PAIRS:
            o_all.append(o[ok]); m_all.append(m[ok])
    if o_all:
        o, m = pd.concat(o_all), pd.concat(m_all)
        res.append({"convention": how, "n": len(o), "r": float(np.corrcoef(o, m)[0, 1]),
                    "rmse": float(np.sqrt(((m - o) ** 2).mean())),
                    "bias": float(m.mean() - o.mean()), "obs_std": float(o.std())})
conv = pd.DataFrame(res).set_index("convention")
display(conv.round(4))

BEST, PRIOR, MARGIN = conv["rmse"].idxmin(), "mean of previous 24 h", 0.05
print(f"\nLowest RMSE: '{BEST}' ({conv.loc[BEST, 'rmse']:.3f} m, r {conv.loc[BEST, 'r']:.3f}); "
      f"spread across conventions {100 * (conv['rmse'].max() / conv['rmse'].min() - 1):.1f}% of RMSE")
CHOSEN = BEST
if PRIOR in conv.index and BEST != PRIOR:
    gain = 1 - conv.loc[BEST, "rmse"] / conv.loc[PRIOR, "rmse"]
    if gain < MARGIN:
        CHOSEN = PRIOR
        print(f"'{BEST}' beats '{PRIOR}' by only {100 * gain:.1f}% - within noise; using "
              f"'{PRIOR}', which the empty first frame implies physically.")
    else:
        print(f"'{BEST}' beats '{PRIOR}' by {100 * gain:.1f}% - the data overrule the prior.")
# instantaneous for elev/u/v (no `averaged` attribute)
OBS_CONVENTION = {"Hs_out": CHOSEN, "Tz_out": CHOSEN, "elev": "instantaneous"}
print("using:", OBS_CONVENTION)

---
## 7. Skill metrics

- **bias** $=\overline{m}-\overline{o}$; **RMSE**; **uRMSE**, the centred RMSE
  (the part not explained by bias)
- **r** Pearson correlation; **SI** scatter index, RMSE / observed mean
- **d** Willmott index of agreement (1 = perfect)
- **slope** of a least-squares fit of model on observation, and **std_ratio**
  $\sigma_m/\sigma_o$ (< 1: the model under-varies)

In [ ]:
SKILL_KEYS = ("n", "obs_mean", "mod_mean", "bias", "rmse", "urmse", "mae", "r", "si",
              "d", "slope", "std_ratio")


def skill(o, m):
    o = np.asarray(o, float); m = np.asarray(m, float)
    ok = np.isfinite(o) & np.isfinite(m)
    o, m = o[ok], m[ok]
    n = o.size
    if n < MIN_PAIRS:
        return {k: np.nan for k in SKILL_KEYS} | {"n": n}
    bias = m.mean() - o.mean()
    rmse = np.sqrt(((m - o) ** 2).mean())
    denom = (np.abs(m - o.mean()) + np.abs(o - o.mean())) ** 2
    return {"n": n, "obs_mean": o.mean(), "mod_mean": m.mean(), "bias": bias,
            "rmse": rmse, "urmse": np.sqrt((((m - m.mean()) - (o - o.mean())) ** 2).mean()),
            "mae": np.abs(m - o).mean(), "r": np.corrcoef(o, m)[0, 1],
            "si": rmse / o.mean() if o.mean() else np.nan,
            "d": 1 - ((m - o) ** 2).sum() / denom.sum() if denom.sum() else np.nan,
            "slope": np.polyfit(o, m, 1)[0],
            "std_ratio": m.std() / o.std() if o.std() else np.nan}


def paired(mod_var, obs_col, how=None, modsel=None):
    """{station: DataFrame(obs, mod)} on the model time axis."""
    modsel = MOD if modsel is None else modsel
    how = how or OBS_CONVENTION.get(mod_var, "instantaneous")
    out = {}
    if mod_var not in modsel:
        return out
    for c_ in STN:
        if obs_col not in OBS[c_]:
            continue
        o = obs_at(OBS[c_][obs_col], mtimes, how)
        m = pd.Series(modsel[mod_var].sel(station=c_).values, index=mtimes)
        df = pd.DataFrame({"obs": o, "mod": m}).dropna()
        if len(df) >= MIN_PAIRS:
            out[c_] = df
    return out


PAIRED = {mv: paired(mv, oc) for mv, oc in PAIRS.items()}
for mv, d in PAIRED.items():
    print(f"{mv:<8} vs {PAIRS[mv]:<5}: {len(d)} stations, {sum(len(v) for v in d.values())} paired days")

tables = {}
for mv, per_station in PAIRED.items():
    if not per_station:
        continue
    t = pd.DataFrame({c_: skill(df["obs"], df["mod"]) for c_, df in per_station.items()}).T
    t = t.sort_values("rmse")
    t.loc["** ALL **"] = skill(pd.concat([d["obs"] for d in per_station.values()]),
                               pd.concat([d["mod"] for d in per_station.values()]))
    tables[mv] = t
    print(f"\n=== {mv} vs {PAIRS[mv]}  [{UNITS[mv]}] ({OBS_CONVENTION.get(mv, 'instantaneous')}) ===")
    display(t[["n", "obs_mean", "mod_mean", "bias", "rmse", "urmse", "r", "si", "d",
               "slope", "std_ratio"]].round(3))

---
## 8. Figures

Shared helpers for time-series panels: months on the axis, 10-min observations
as a light background layer, matched observations in ink, model in the run
colour. Station numbers (Fig. 1) are reused by the Taylor diagram, the bias map
and the cross-check.

In [ ]:
def fmt_stats(s, unit_fmt=".2f"):
    return (f"bias {fs.signed(s['bias'], unit_fmt)} · RMSE {fs.unsigned(s['rmse'], unit_fmt)}"
            f" · r {fs.unsigned(s['r'])}")


def month_axis(ax, show_labels=True):
    ax.set_xlim(pd.Timestamp(f"{YEAR}-01-01"), pd.Timestamp(f"{YEAR + 1}-01-01"))
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_minor_locator(mticker.NullLocator())
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(
        lambda x, _: mdates.num2date(x).strftime("%b")[0] if show_labels else ""))


def ts_panel(ax, c_, mod_var, obs_col, pairs, show_raw=True):
    """Model (daily), matched observations, and the 10-min record behind them."""
    df = pairs[c_]
    if show_raw and obs_col in OBS[c_]:
        raw = OBS[c_][obs_col].dropna()
        if len(raw):
            raw = raw.reindex(pd.date_range(raw.index.min(), raw.index.max(), freq="10min"))
            ax.plot(raw.index, raw.values, color=RAW_COLOUR, lw=0.3, zorder=1, rasterized=True)
    m_full = pd.Series(MOD[mod_var].sel(station=c_).values, index=mtimes)
    o_full = df["obs"].reindex(mtimes)
    ax.plot(o_full.index, o_full.values, color=fs.OBS_COLOUR, lw=0.8, zorder=2)
    ax.plot(m_full.index, m_full.values, color=MODEL_COLOUR, lw=0.9, zorder=3)
    s = skill(df["obs"], df["mod"])
    fs.stats_line(ax, fmt_stats(s) + f" \u00b7 n {s['n']:.0f}", ha="right")
    fs.stats_line(ax, short_name(c_), ha="left", fontsize=6.5, color=fs.INK,
                  fontweight="bold")
    ax.set_ylim(bottom=0 if mod_var != "elev" else None)
    ax.yaxis.set_major_locator(mticker.MaxNLocator(3))
    return pd.DataFrame({"station": c_, "time": mtimes, "model": m_full.values,
                         "observed_matched": o_full.values})


def ts_legend(fig, mod_var, obs_col):
    how = OBS_CONVENTION.get(mod_var, "instantaneous")
    matched = "observed, 24 h mean" if "24 h" in how else "observed at 00:00"
    handles = [Line2D([], [], color=MODEL_COLOUR, lw=0.9, label=f"GETM {RUN_NAME}"),
               Line2D([], [], color=fs.OBS_COLOUR, lw=0.8, label=matched),
               Line2D([], [], color=RAW_COLOUR, lw=0.8, label="observed, 10 min")]
    fig.legend(handles=handles, loc="outside upper center", ncol=3)


# supplementary pages run from the most to the least energetic station
ENERGY = pd.Series({c_: d["obs"].mean() for c_, d in PAIRED.get("Hs_out", {}).items()})
ORDER = list(ENERGY.sort_values(ascending=False).index)
ORDER += [c_ for c_ in STN if c_ not in ORDER]

### Fig. 2 — Hs and Tp at the four best-performing stations

**Selection.** Among the stations with at least `KEY_MIN_DAYS` paired days for
both Hs and Tp, every station is scored by its normalised RMSE,
RMSE / σ_obs, for each variable. That single number folds together bias,
correlation and variance (RMSE² = bias² + centred RMSD²), and below 1 the model
beats a constant equal to the observed mean. A station's score is the **worse**
of its Hs and Tp values, so it must do well on both; scores within 0.01 count
as tied and the lower mean of the two decides. The four best are the key
stations. The ranking is printed below and saved as `key_station_ranking.csv`;
`KEY_STATIONS` in the configuration fixes the choice.

"Best" is relative for Tp: in 2015 every station has a Tp nRMSE above 1 and a
Tp correlation below 0.2, so even the key stations show little day-to-day Tp
skill. They are the stations where Hs is good and Tp is least poor.

The 24 h mean observation is the one the model is scored against; the 10-min
record shows what that mean smooths over.

In [ ]:
def key_station_ranking(min_days=KEY_MIN_DAYS):
    """Normalised RMSE per station for Hs and Tp; score = the worse of the two."""
    hs, tp = PAIRED.get("Hs_out", {}), PAIRED.get("Tz_out", {})
    rows = []
    for c_ in hs:
        if c_ not in tp:
            continue
        row = {"station": c_, "label": short_name(c_)}
        for key, d in (("hs", hs[c_]), ("tp", tp[c_])):
            e = d["mod"] - d["obs"]
            row[f"n_{key}"] = len(d)
            row[f"nrmse_{key}"] = float(np.sqrt((e ** 2).mean()) / d["obs"].std())
            row[f"bias_{key}"] = float(e.mean())
            row[f"r_{key}"] = float(np.corrcoef(d["obs"], d["mod"])[0, 1])
        rows.append(row)
    tab = pd.DataFrame(rows).set_index("station")
    tab["eligible"] = (tab["n_hs"] >= min_days) & (tab["n_tp"] >= min_days)
    tab["score"] = tab[["nrmse_hs", "nrmse_tp"]].max(axis=1)
    tab["mean_nrmse"] = tab[["nrmse_hs", "nrmse_tp"]].mean(axis=1)
    tab["_score2"] = tab["score"].round(2)          # within 0.01 counts as a tie
    tab = tab.sort_values(["eligible", "_score2", "mean_nrmse"], ascending=[False, True, True])
    tab["rank"] = np.where(tab["eligible"], np.arange(1, len(tab) + 1), np.nan)
    return tab.drop(columns="_score2")


RANKING = key_station_ranking()
RANKING.round(4).to_csv(OUT_DIR / "key_station_ranking.csv")
display(RANKING.round(3))
KEY = list(KEY_STATIONS) if KEY_STATIONS else list(RANKING.index[RANKING["eligible"]][:4])
print("key stations:", ", ".join(short_name(c_) for c_ in KEY))
fig, axes = fs.figure("double", 20 + 30 * len(KEY), nrows=len(KEY), ncols=2, squeeze=False,
                      sharex=True)
csv, letters = [], iter("abcdefghijklmnop")
for r_, c_ in enumerate(KEY):
    for k, mv in enumerate(["Hs_out", "Tz_out"]):
        ax = axes[r_, k]
        csv.append(ts_panel(ax, c_, mv, PAIRS[mv], PAIRED[mv]).assign(variable=mv))
        month_axis(ax, r_ == len(KEY) - 1)
        ax.set_ylabel(f"{LABEL[mv]} ({UNITS[mv]})")
ts_legend(fig, "Hs_out", "hm0")
for ax in axes.flat:
    fs.panel_label(ax, next(letters))
fs.save_figure(fig, "fig02_wave_timeseries_key", FIG_DIR, data=pd.concat(csv, ignore_index=True))

### Fig. 3 — pooled model vs observed

All stations and days together, as a density of pairs (log colour scale) with
the 1:1 line (dashed) and an ordinary least-squares fit (grey).

In [ ]:
have = [mv for mv in ("Hs_out", "Tz_out") if PAIRED.get(mv)]
fig, axes = fs.figure("onehalf", 60, ncols=len(have), squeeze=False)
csv, hb = [], None
for k, (ax, mv) in enumerate(zip(axes[0], have)):
    o = pd.concat([d["obs"] for d in PAIRED[mv].values()])
    m = pd.concat([d["mod"] for d in PAIRED[mv].values()])
    hi = float(np.nanpercentile(np.r_[o, m], 99.9)) * 1.05
    lo = 0.0
    hb = ax.hexbin(o, m, gridsize=40, extent=(lo, hi, lo, hi), cmap=fs.cmap_density(),
                   norm=mcolors.LogNorm(vmin=1), mincnt=1, linewidths=0.1, edgecolors="face",
                   rasterized=True, zorder=2)
    ax.plot([lo, hi], [lo, hi], ls=(0, (4, 2)), lw=0.6, color=fs.INK, zorder=3)
    b, a = np.polyfit(o, m, 1)
    ax.plot([lo, hi], [a + b * lo, a + b * hi], color=fs.INK2, lw=1.0, zorder=4)
    s = skill(o, m)
    fs.stats_line(ax, f"n {s['n']:,} · {fmt_stats(s)}\nSI {s['si']:.2f} · slope {b:.2f}")
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi); ax.set_box_aspect(1)
    ax.set_xlabel(f"Observed {OBS_LABEL[PAIRS[mv]]} ({UNITS[mv]})")
    ax.set_ylabel(f"Modelled {LABEL[mv]} ({UNITS[mv]})")
    fs.panel_label(ax, "ab"[k])
    csv.append(pd.DataFrame({"variable": mv, "observed": o.values, "modelled": m.values}))
cb = fig.colorbar(hb, ax=axes[0], shrink=0.75, aspect=18, pad=0.02)
cb.set_label("Station-days per bin"); cb.outline.set_linewidth(0.4)
cb.ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:g}"))
cb.ax.tick_params(which="both", width=0.4)
handles = [Line2D([], [], ls=(0, (4, 2)), lw=0.6, color=fs.INK, label="1:1"),
           Line2D([], [], color=fs.INK2, lw=1.0, label="least-squares fit")]
axes[0, 0].legend(handles=handles, loc="upper left")
fs.save_figure(fig, "fig03_wave_scatter", FIG_DIR, data=pd.concat(csv, ignore_index=True))

### Fig. 4 — Taylor diagram for Hs

Angle = correlation, radius = model standard deviation normalised by the
observed one; the reference (star) is a perfect match. Grey arcs are centred
RMS differences (normalised). A Taylor diagram says nothing about bias, so the
marker colour carries it. Numbers refer to the station key (as in Fig. 1).
Variables are chosen with `TAYLOR_VARS`.

In [ ]:
def taylor(ax, stats, cmap, norm):
    """Taylor diagram on a polar axes; extends to 180 deg if any r < 0."""
    stats = stats[np.isfinite(stats["r"]) & np.isfinite(stats["std_ratio"])]
    rmax = max(1.5, np.ceil(stats["std_ratio"].max() * 1.08 / 0.25) * 0.25)
    half = bool((stats["r"] < 0).any())
    tmax = np.pi if half else np.pi / 2
    ax.set_thetamin(0); ax.set_thetamax(np.degrees(tmax))
    ax.set_rlim(0, rmax)
    ax.grid(False)
    corr = [0.2, 0.4, 0.6, 0.8, 0.9, 0.95, 0.99]
    if half:
        corr = [-0.99, -0.9, -0.8, -0.6, -0.4, -0.2, 0.0] + corr
    ax.set_thetagrids(np.degrees(np.arccos(corr)), labels=[fs.unsigned(c, "g") for c in corr])
    step = 0.5 if rmax <= 2.5 else 1.0
    ax.set_rticks(np.arange(0, rmax + 1e-9, step))
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:g}"))
    ax.tick_params(axis="both", which="both", labelsize=6.5)
    ax.xaxis.set_tick_params(pad=1)
    for c in corr:
        if abs(c) > 0:
            ax.plot([np.arccos(c)] * 2, [0, rmax], color=fs.HAIRLINE, lw=0.4, zorder=0)
    th = np.linspace(0, tmax, 300)
    ax.plot(th, np.ones_like(th), color=fs.INK3, lw=0.5, ls=(0, (4, 2)), zorder=1)
    phi = np.linspace(0, np.pi, 400)
    arcs = (0.25, 0.5, 0.75, 1.0) if rmax <= 2.5 else tuple(e for e in (0.5, 1, 2, 3, 4) if e < rmax)
    for e in arcs:
        x, y = 1 + e * np.cos(phi), e * np.sin(phi)
        rr, tt = np.hypot(x, y), np.arctan2(y, x)
        keep = (rr <= rmax) & (tt <= tmax)
        ax.plot(tt[keep], rr[keep], color="#bdbab1", lw=0.5, zorder=1)
        k = np.argmin(np.abs(phi - 2.2))
        if keep[k]:
            ax.text(tt[k], rr[k], f"{e:g}", fontsize=5.5, color=fs.INK3, ha="center",
                    va="center", zorder=2, path_effects=fs.halo(1.2))
    ax.plot(0, 1, marker="*", ms=6, color=fs.INK, zorder=6, clip_on=False)
    theta = np.arccos(np.clip(stats["r"].to_numpy(float), -1, 1))
    rad = stats["std_ratio"].to_numpy(float)
    sc = ax.scatter(theta, rad, c=stats["bias"], cmap=cmap, norm=norm, s=16,
                    edgecolors=fs.INK, linewidths=0.35, zorder=5)
    if half:
        ax.text(np.pi / 2, rmax * 1.2, "Correlation", ha="center", va="bottom", fontsize=7)
    else:
        ax.text(np.pi / 4, rmax * 1.16, "Correlation", rotation=-45, ha="center",
                va="center", fontsize=7)
    ax.annotate("Normalised standard deviation", xy=(0.75, 0.25) if half else (0.5, 0.0),
                xycoords="axes fraction", xytext=(0, -11), textcoords="offset points",
                ha="center", va="top", fontsize=7)
    return sc, theta, rad, half


have = [mv for mv in TAYLOR_VARS if PAIRED.get(mv)]
STATS_T = {}
for mv in have:
    st = pd.DataFrame({c_: skill(d["obs"], d["mod"]) for c_, d in PAIRED[mv].items()}).T
    STATS_T[mv] = st[np.isfinite(st["r"])]
in_key = sorted(set().union(*[STATS_T[mv].index for mv in have]), key=NUMBER.get)

# Explicit layout (mm). Matplotlib keeps polar axes square, and a half circle
# only fills the middle half of that square, so constrained layout would leave
# dead space; here every wedge is exactly A tall. One diagram: key on the right;
# several: key underneath.
halfs = [bool((STATS_T[mv]["r"] < 0).any()) for mv in have]
f = [2 if h else 1 for h in halfs]
single = len(have) == 1
top, below, cbar_h = 15.0, 12.0, 2.4
if single:
    top = 10.0                         # no panel letter to make room for
    left, right, gap = 7.0, 2.0, 0.0
    A = 62.0 if f[0] == 1 else 50.0
    key_w = 36.0
    W = left + f[0] * A + 18.0 + key_w + right
    H = top + A + below + cbar_h + 11.0
else:
    W, left, right, gap = 183.0, 7.0, 11.0, 24.0
    A = min((W - left - right - gap * (len(have) - 1)) / sum(f), 62.0)
    key_h = 2.8 * int(np.ceil(len(in_key) / 4)) + 1
    H = top + A + below + cbar_h + 9.0 + key_h + 2.0
fig = plt.figure(figsize=(W * fs.MM, H * fs.MM))
fig.set_layout_engine("none")      # fixed positions; "none" at creation does not survive savefig
wedge_bottom = H - top - A
csv, labelled, x0 = [], [], left
for k, mv in enumerate(have):
    side = f[k] * A
    ax = fig.add_axes([x0 / W, (wedge_bottom - (side - A) / 2) / H, side / W, side / H],
                      projection="polar")
    st = STATS_T[mv]
    lim = fs.nice_limit(st["bias"], q=100)
    sc, theta, rad, half = taylor(ax, st, fs.cmap_diverging(), mcolors.Normalize(-lim, lim))
    cw = min(0.8 * A, 48.0)
    cax = fig.add_axes([(x0 + side / 2 - cw / 2) / W, (wedge_bottom - below - cbar_h) / H,
                        cw / W, cbar_h / H])
    fs.diverging_colorbar(fig, sc, cax=cax, orientation="horizontal",
                          ticks=np.linspace(-lim, lim, 5),
                          label=f"{LABEL[mv]} bias, model \u2212 observed ({UNITS[mv]})",
                          low_text="low", high_text="high")
    if not single:
        fig.text((x0 - 5) / W, (H - 3) / H, "ab"[k], ha="left", va="top", fontsize=8,
                 fontweight="bold")
    labelled.append((ax, theta, rad, [str(NUMBER[c_]) for c_ in st.index]))
    csv.append(st.assign(variable=mv, number=[NUMBER[c_] for c_ in st.index],
                         label=[short_name(c_) for c_ in st.index]))
    x0 += side + gap
if single:
    key_ax = fig.add_axes([(W - right - key_w) / W, 3.0 / H, key_w / W, (H - top - 1.0) / H])
    station_key(key_ax, in_key, ncols=1)
else:
    station_key(fig.add_axes([left / W, 1.5 / H, (W - left - right) / W, key_h / H]), in_key)
for ax, theta, rad, labs in labelled:
    fs.place_labels(ax, theta, rad, labs, fontsize=5.5, radii=(3, 5.5, 8.5), stay_inside=False,
                    leader_from=8)
fs.save_figure(fig, "fig04_wave_taylor", FIG_DIR,
               data=pd.concat(csv).rename_axis("station").reset_index())

### Fig. 5 — station bias on the map

Mean bias per station (model − observed) for Hs (top) and Tp (bottom), on a
symmetric diverging scale centred on zero. The zoom panels show the Amelander
Zeegat array, whose paired buoys are nudged apart (hairlines lead to the true
positions). Station numbers as in Fig. 1.

In [ ]:
have = [mv for mv in ("Hs_out", "Tz_out") if PAIRED.get(mv)]
shown = sorted(set().union(*[PAIRED[mv].keys() for mv in have]), key=NUMBER.get)
fig, axes, key_ax = map_figure(len(have), extra_mm=12, key_lines=int(np.ceil(len(shown) / 4)))
csv, pending = [], []
for r_, mv in enumerate(have):
    st = pd.DataFrame({c_: skill(d["obs"], d["mod"]) for c_, d in PAIRED[mv].items()}).T
    st = st[np.isfinite(st["bias"])].join(match[["lon", "lat", "label"]])
    lim = fs.nice_limit(st["bias"], q=100)
    norm, cmap = mcolors.Normalize(-lim, lim), fs.cmap_diverging()
    ax_a, ax_b = axes[r_]
    last = r_ == len(have) - 1
    wave_map(ax_a, MAP_EXTENT, labels=(last, True))
    wave_map(ax_b, ZOOM_EXTENT, zoom=True, labels=(last, False))
    zoom_letter(ax_a, "bd"[r_] if len(have) > 1 else "b")
    z = st.index.str.startswith(ZOOM_PREFIX)
    kw = dict(cmap=cmap, norm=norm, edgecolor=fs.INK, linewidth=0.4)
    sc = ax_a.scatter(st.loc[~z, "lon"], st.loc[~z, "lat"], c=st.loc[~z, "bias"], s=22,
                      zorder=8, **kw)
    _, zx, zy = plot_station_markers(ax_b, st.loc[z, "lon"], st.loc[z, "lat"], repel=True,
                                     c=st.loc[z, "bias"], s=22, **kw)
    fs.diverging_colorbar(fig, sc, ax=axes[r_, :], ticks=np.linspace(-lim, lim, 5),
                          label=f"{LABEL[mv]} bias ({UNITS[mv]})", shrink=0.9, aspect=16,
                          pad=0.02)
    fs.corner_note(ax_a, LABEL[mv], corners=("upper left",), fontweight="bold",
                   path_effects=fs.halo())
    pending.append((ax_a, st.loc[~z], ax_b, zx, zy, st.loc[z]))
    csv.append(st[["label", "lon", "lat", "n", "obs_mean", "mod_mean", "bias", "rmse", "r"]]
               .assign(variable=mv))
station_key(key_ax, shown)
for ax, l in zip(axes.flat, "abcd"):
    fs.panel_label(ax, l)
for ax_a, sa, ax_b, zx, zy, sb in pending:
    fs.place_labels(ax_a, sa["lon"], sa["lat"], [str(NUMBER[c_]) for c_ in sa.index], fontsize=6)
    fs.place_labels(ax_b, zx, zy, [str(NUMBER[c_]) for c_ in sb.index], fontsize=6)
fs.save_figure(fig, "fig05_wave_bias_map", FIG_DIR,
               data=pd.concat(csv).rename_axis("station").reset_index())

### Fig. 6 — does the Hs error depend on the sea state?

All stations pooled, binned by observed Hs. (a) Mean bias (dot) and the
interquartile range of the daily error (bar); (b) RMSE. The number of
station-days per bin is given under each bin label.

In [ ]:
o = pd.concat([d["obs"] for d in PAIRED["Hs_out"].values()])
m = pd.concat([d["mod"] for d in PAIRED["Hs_out"].values()])
edges = np.array([0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 10.0])
rows = []
for k in range(len(edges) - 1):
    sel = (o >= edges[k]) & (o < edges[k + 1])
    if sel.sum() < 10:
        continue
    e = (m[sel] - o[sel]).to_numpy()
    hi_lab = f"{edges[k + 1]:g}" if edges[k + 1] < 10 else ""
    rows.append({"bin": f"{edges[k]:g}–{hi_lab}" if hi_lab else f"≥{edges[k]:g}",
                 "lo": edges[k], "hi": edges[k + 1], "n": int(sel.sum()), "bias": e.mean(),
                 "q25": np.percentile(e, 25), "q75": np.percentile(e, 75),
                 "rmse": np.sqrt((e ** 2).mean())})
cond = pd.DataFrame(rows)
x = np.arange(len(cond))
fig, (ax1, ax2) = fs.figure("onehalf", 86, nrows=2, sharex=True)
ax1.axhline(0, color=fs.INK, lw=0.5)
ax1.vlines(x, cond["q25"], cond["q75"], color=MODEL_COLOUR, alpha=0.45, lw=2.6)
ax1.plot(x, cond["bias"], "o", ms=3.4, color=MODEL_COLOUR, zorder=3)
ax1.set_ylabel("Hs error, model − observed (m)")
ax2.bar(x, cond["rmse"], width=0.62, color=MODEL_COLOUR)
ax2.set_ylabel("Hs RMSE (m)")
ticklabels = [f"{b}\n{n:,}" for b, n in zip(cond["bin"], cond["n"])]
ax2.set_xticks(x, ticklabels, fontsize=6.5)
ax2.xaxis.set_minor_locator(mticker.NullLocator())
ax2.set_xlabel("Observed Hs (m) and number of station-days")
for ax, l in ((ax1, "a"), (ax2, "b")):
    fs.panel_label(ax, l)
handles = [Line2D([], [], ls="", marker="o", ms=3.4, color=MODEL_COLOUR, label="mean bias"),
           Line2D([], [], color=MODEL_COLOUR, alpha=0.45, lw=2.6, label="interquartile range")]
ax1.legend(handles=handles, loc="lower left")
fs.save_figure(fig, "fig06_wave_bias_by_sea_state", FIG_DIR, data=cond)

### Supplementary — every station

Hs and Tp time series (paginated, `PER_PAGE` stations per page) and one scatter
panel per station. Scatter panels are scaled individually: the Wadden wave
climate spans two orders of magnitude, and a shared axis would collapse the
sheltered stations into a corner. Stations are ordered from most to least
energetic. The CSV twins hold the daily model and matched observations; the
10-min background series is the input file `processed/csv/<station>.csv.gz`.

In [ ]:
def timeseries_pages(mod_var, tag):
    pairs = PAIRED.get(mod_var, {})
    if not pairs:
        print(f"no pairs for {mod_var}")
        return
    stns = [c_ for c_ in ORDER if c_ in pairs]
    npages = int(np.ceil(len(stns) / PER_PAGE))
    for p in range(npages):
        chunk = stns[p * PER_PAGE:(p + 1) * PER_PAGE]
        fig, axes = fs.figure("double", 16 + 24 * len(chunk), nrows=len(chunk), sharex=True,
                              squeeze=False)
        csv = []
        for k, (ax, c_) in enumerate(zip(axes[:, 0], chunk)):
            csv.append(ts_panel(ax, c_, mod_var, PAIRS[mod_var], pairs))
            month_axis(ax, k == len(chunk) - 1)
            ax.set_ylabel(f"{LABEL[mod_var]} ({UNITS[mod_var]})")
        ts_legend(fig, mod_var, PAIRS[mod_var])
        name = f"{tag}_p{p + 1}" if npages > 1 else tag
        fs.save_figure(fig, name, FIG_DIR, data=pd.concat(csv, ignore_index=True), close=True)


def scatter_grid(mod_var, tag, ncols=5):
    pairs = PAIRED.get(mod_var, {})
    if not pairs:
        return
    stns = [c_ for c_ in ORDER if c_ in pairs]
    nrows = int(np.ceil(len(stns) / ncols))
    fig, axes = fs.figure("double", 12 + 37 * nrows, nrows=nrows, ncols=ncols, squeeze=False)
    csv, unit = [], UNITS[mod_var]
    for k, c_ in enumerate(stns):
        ax, df = axes[k // ncols, k % ncols], pairs[c_]
        lo = float(min(df["obs"].min(), df["mod"].min()))
        hi = float(max(df["obs"].max(), df["mod"].max()))
        pad = 0.05 * (hi - lo) or 0.05
        lim = (max(lo - pad, 0) if mod_var != "elev" else lo - pad, hi + pad)
        ax.plot(lim, lim, ls=(0, (4, 2)), lw=0.6, color=fs.INK, zorder=1)
        ax.scatter(df["obs"], df["mod"], s=3, color=MODEL_COLOUR, alpha=0.5, linewidths=0,
                   rasterized=True, zorder=2)
        ax.set_xlim(lim); ax.set_ylim(lim); ax.set_box_aspect(1)
        ax.xaxis.set_major_locator(mticker.MaxNLocator(3))
        ax.yaxis.set_major_locator(mticker.MaxNLocator(3))
        s = skill(df["obs"], df["mod"])
        ax.set_title(short_name(c_), fontsize=6.5, pad=2)
        fs.corner_note(ax, f"bias {fs.signed(s['bias'])}\nRMSE {s['rmse']:.2f}\nr {fs.unsigned(s['r'])}",
                       df["obs"], df["mod"], corners=("upper left", "lower right"), fontsize=6)
        if k // ncols == nrows - 1 or k + ncols >= len(stns):
            ax.set_xlabel(f"Observed ({unit})")
        if k % ncols == 0:
            ax.set_ylabel(f"Modelled ({unit})")
        csv.append(df.assign(station=c_).rename_axis("time").reset_index())
    for k in range(len(stns), nrows * ncols):
        axes[k // ncols, k % ncols].axis("off")
    fs.save_figure(fig, tag, FIG_DIR, data=pd.concat(csv, ignore_index=True), close=True)


timeseries_pages("Hs_out", "figS1_wave_timeseries_hs")
timeseries_pages("Tz_out", "figS2_wave_timeseries_tp")
scatter_grid("Hs_out", "figS3_wave_scatter_hs_by_station")
scatter_grid("Tz_out", "figS4_wave_scatter_tp_by_station")

---
## 9. Is the bias a cell-selection artefact, or the model?

At 500 m a station can sit in a cell that is hydrodynamically unlike its real
surroundings — a channel cell instead of a flat, or vice versa. **The
reachable-bias test**: for every cell in the `(2R+1)×(2R+1)` window compute the
mean Hs bias against the same observation, and ask whether that range contains
zero.

- **Yes** → some cell within ~2.5 km would have had no mean bias: a
  *representativeness* problem.
- **No** → every nearby cell errs the same way; no matching rule can fix it and
  the discrepancy belongs to the model or its forcing.

Three selection rules are compared, all decided independently of skill:

| rule | idea | risk |
|---|---|---|
| `nearest` | closest wet cell | may land in the wrong feature |
| `depth-matched` | cell whose depth is closest to the station's | only as good as the station depth |
| `window-median` | median over the window | smooths real gradients, least arbitrary |

> **Do not pick the cell that maximises skill.** With 25 cells to choose from
> almost any model can be made to look good. Choose the rule *a priori* and
> report it. The station depths used by `depth-matched` come from the EMODnet
> DTM, which is unreliable on exactly the tidal flats where the bias is worst.

In [ ]:
_bed = ov.set_index("code").reindex(STN)["bed_level_nap_m"].to_numpy(float)
TARGET_DEPTH = -_bed
strategies = {"nearest": select_cell(MODW, "nearest"),
              "depth-matched": select_cell(MODW, "depth-matched", target_depth=TARGET_DEPTH),
              "window-median": select_cell(MODW, "window-median")}
summary = {}
for name, sel_ in strategies.items():
    t = pd.DataFrame({c_: skill(d["obs"], d["mod"])
                      for c_, d in paired("Hs_out", PAIRS["Hs_out"], modsel=sel_).items()}).T
    w = t["n"] / t["n"].sum()
    summary[name] = {"stations": len(t), "mean |bias|": float((t["bias"].abs() * w).sum()),
                     "mean bias": float((t["bias"] * w).sum()),
                     "mean RMSE": float((t["rmse"] * w).sum()),
                     "mean r": float((t["r"] * w).sum()), "median slope": float(t["slope"].median())}
display(pd.DataFrame(summary).T.round(3))


def reachable_bias(mod_var="Hs_out", obs_col=PAIRS["Hs_out"]):
    how = OBS_CONVENTION.get(mod_var, "instantaneous")
    field = MODW[mod_var]
    rows, percell = [], {}
    for c_ in STN:
        if obs_col not in OBS[c_]:
            continue
        o = obs_at(OBS[c_][obs_col], mtimes, how)
        if o.notna().sum() < MIN_PAIRS:
            continue
        cell = field.sel(station=c_).values
        b = np.full(field.sizes["win"], np.nan)
        for w in range(field.sizes["win"]):
            pair = pd.DataFrame({"o": o, "m": pd.Series(cell[:, w], index=mtimes)}).dropna()
            if len(pair) >= MIN_PAIRS:
                b[w] = pair["m"].mean() - pair["o"].mean()
        if not np.isfinite(b).any():
            continue
        percell[c_] = b
        lo_, hi_ = float(np.nanmin(b)), float(np.nanmax(b))
        rows.append({"code": c_, "label": short_name(c_), "n_cells": int(np.isfinite(b).sum()),
                     "obs_mean": float(o.dropna().mean()), "bias_nearest": float(b[CENTRE]),
                     "bias_min": lo_, "bias_max": hi_,
                     "bias_best": float(b[np.nanargmin(np.abs(b))]),
                     "reaches_zero": bool(lo_ <= 0 <= hi_)})
    tab = pd.DataFrame(rows).set_index("code")
    tab["removable"] = np.where(tab["bias_nearest"].abs() > 0.05,
                                1 - tab["bias_best"].abs() / tab["bias_nearest"].abs().replace(0, np.nan),
                                np.nan)
    return tab.sort_values("bias_nearest", key=abs, ascending=False), percell


REACH, PERCELL = reachable_bias()
display(REACH.round(3))
nz = int(REACH["reaches_zero"].sum())
print(f"\n{nz} of {len(REACH)} stations have SOME cell in the window that removes the mean bias.")
stuck = REACH[~REACH["reaches_zero"]]
if len(stuck):
    print(f"{len(stuck)} stations cannot be fixed by ANY cell (smallest achievable |bias| "
          f"{stuck['bias_best'].abs().min():.2f}-{stuck['bias_best'].abs().max():.2f} m): "
          + ", ".join(stuck["label"]))
REACH.round(4).to_csv(OUT_DIR / f"cell_reachable_bias_{YEAR}.csv")
pd.DataFrame(summary).T.round(4).to_csv(OUT_DIR / f"cell_strategy_{YEAR}.csv")

### Fig. 7 — could a different model cell have removed the Hs bias?

Bar: range of mean bias over the cells of the window. Dot: the nearest cell
(what is used). Tick: the best cell in the window. Where the bar does not cross
zero (grey), no cell within ~2.5 km agrees with the observation on average.

In [ ]:
t = REACH.sort_values("bias_nearest")
y = np.arange(len(t))
fig, ax = fs.figure("onehalf", 22 + 3.2 * len(t))
ok_col, stuck_col = mcolors.to_rgba(MODEL_COLOUR, 0.35), mcolors.to_rgba(fs.INK3, 0.55)
ax.axvline(0, color=fs.INK, lw=0.5, zorder=1)
ax.hlines(y, t["bias_min"], t["bias_max"], lw=3.2, zorder=2,
          colors=[ok_col if z else stuck_col for z in t["reaches_zero"]])
ax.plot(t["bias_nearest"], y, "o", ms=3.2, color=MODEL_COLOUR, zorder=4)
ax.plot(t["bias_best"], y, "|", ms=5.5, mew=0.9, color=fs.INK, zorder=5)
ax.set_yticks(y, t["label"], fontsize=6)
ax.yaxis.set_minor_locator(mticker.NullLocator())
ax.tick_params(axis="y", length=0)
ax.spines["left"].set_visible(False)
ax.set_ylim(-0.7, len(t) - 0.3)
ax.set_xlabel("Hs bias, model − observed (m)")
w = 2 * NEIGHBOURHOOD_R + 1
handles = [Line2D([], [], color=ok_col, lw=3.2, label=f"range over {w}×{w} window"),
           Line2D([], [], color=stuck_col, lw=3.2, label="range excludes zero"),
           Line2D([], [], ls="", marker="o", ms=3.2, color=MODEL_COLOUR, label="nearest cell"),
           Line2D([], [], ls="", marker="|", ms=5.5, mew=0.9, color=fs.INK, label="best cell")]
fig.legend(handles=handles, loc="outside upper center", ncol=2)
fs.save_figure(fig, "fig07_wave_reachable_bias", FIG_DIR,
               data=t[["label", "n_cells", "obs_mean", "bias_nearest", "bias_min", "bias_max",
                       "bias_best", "reaches_zero"]])

### Does the cell chosen for Hs also help Tp?

Choose the cell on **Hs**, then check what it does to **Tp**, which had no say
in the choice. If Tp improves too and both variables prefer the same cell, the
nearest cell really was misplaced. If Tp gets no better and the preferred cells
are unrelated, minimising the Hs bias is fitting noise: keep `nearest` and treat
the bias as a model error.

In [ ]:
def cross_check(primary=("Hs_out", PAIRS["Hs_out"]), secondary=("Tz_out", PAIRS["Tz_out"])):
    sv, so = secondary
    if sv not in MODW or not PERCELL:
        print(f"{sv} not available - skipping the cross-check")
        return None
    how_s = OBS_CONVENTION.get(sv, "instantaneous")
    dj = np.array([o[0] for o in OFFSETS]); di = np.array([o[1] for o in OFFSETS])
    rows = []
    for c_ in STN:
        if c_ not in PERCELL or so not in OBS[c_]:
            continue
        bp = PERCELL[c_]
        os_ = obs_at(OBS[c_][so], mtimes, how_s)
        if os_.notna().sum() < MIN_PAIRS:
            continue
        sec = MODW[sv].sel(station=c_).values
        bs = np.full(sec.shape[1], np.nan)
        for w in range(sec.shape[1]):
            pair = pd.DataFrame({"o": os_, "m": pd.Series(sec[:, w], index=mtimes)}).dropna()
            if len(pair) >= MIN_PAIRS:
                bs[w] = pair["m"].mean() - pair["o"].mean()
        if not (np.isfinite(bp).any() and np.isfinite(bs).any()):
            continue
        w_hs, w_tz = int(np.nanargmin(np.abs(bp))), int(np.nanargmin(np.abs(bs)))
        if not np.isfinite(bs[w_hs]):
            continue
        rows.append({"code": c_, "label": short_name(c_),
                     "tz_bias_nearest": float(bs[CENTRE]), "tz_bias_at_hs_cell": float(bs[w_hs]),
                     "tz_bias_best": float(bs[w_tz]),
                     "tz_improved": abs(bs[w_hs]) < abs(bs[CENTRE]), "same_cell": w_hs == w_tz,
                     "cell_gap_km": 0.5 * float(np.hypot(dj[w_hs] - dj[w_tz], di[w_hs] - di[w_tz])),
                     "hs_cell_offset_km": 0.5 * float(np.hypot(dj[w_hs], di[w_hs]))})
    return pd.DataFrame(rows).set_index("code") if rows else None


CROSS = cross_check()
if CROSS is not None:
    display(CROSS.round(3))
    n, imp, same = len(CROSS), int(CROSS["tz_improved"].sum()), int(CROSS["same_cell"].sum())
    gain = (CROSS["tz_bias_nearest"].abs() - CROSS["tz_bias_at_hs_cell"].abs()).median()
    print(f"\nTp improves at {imp} of {n} stations when the cell is chosen on Hs "
          f"(median change in |Tp bias| {gain:+.3f} s); Hs and Tp prefer the same cell at "
          f"{same} of {n} (median distance {CROSS['cell_gap_km'].median():.1f} km).")
    if imp > 0.6 * n and same > 0.3 * n:
        print("The choice carries over to a variable that did not influence it: a genuine "
              "representativeness fix rather than curve-fitting.")
    else:
        print("The Hs-optimal cell does NOT systematically help Tp: keep CELL_STRATEGY="
              "'nearest' and treat the Hs bias as a model error.")
    CROSS.round(4).to_csv(OUT_DIR / f"cell_cross_check_{YEAR}.csv")

### Fig. 8 — the cross-check

Each point is a station (numbers as in Fig. 1). Below the 1:1 line the cell
chosen on Hs also reduced the Tp bias; in the shaded area above it, it made Tp
worse.

In [ ]:
if CROSS is not None:
    fig, ax = fs.figure("single", 84)
    x = CROSS["tz_bias_nearest"].abs(); yv = CROSS["tz_bias_at_hs_cell"].abs()
    hi = float(max(x.max(), yv.max())) * 1.1
    ax.fill_between([0, hi], [0, hi], hi, color="#f1efea", lw=0, zorder=0)
    ax.plot([0, hi], [0, hi], ls=(0, (4, 2)), lw=0.6, color=fs.INK, zorder=1)
    ax.scatter(x, yv, s=12, color=MODEL_COLOUR, edgecolor="white", linewidth=0.4, zorder=3)
    ax.text(0.04, 0.96, "Tp worse", transform=ax.transAxes, ha="left", va="top",
            color=fs.INK2, style="italic", fontsize=6.5)
    ax.text(0.96, 0.04, "Tp better", transform=ax.transAxes, ha="right", va="bottom",
            color=fs.INK2, style="italic", fontsize=6.5)
    ax.set_xlim(0, hi); ax.set_ylim(0, hi); ax.set_box_aspect(1)
    ax.set_xlabel("|Tp bias| at the nearest cell (s)")
    ax.set_ylabel("|Tp bias| at the cell chosen on Hs (s)")
    fs.place_labels(ax, x, yv, [str(NUMBER[c_]) for c_ in CROSS.index], fontsize=6)
    fs.save_figure(fig, "fig08_wave_cell_cross_check", FIG_DIR,
                   data=CROSS.assign(abs_tz_bias_nearest=x, abs_tz_bias_at_hs_cell=yv))

---
## 10. Water level — aliased, read with care

Daily 00:00 samples alias M2 to ~14.8 days (see the header). The comparison is
therefore **not** a measure of tidal skill. What it can still show: a datum
offset between the GETM reference level and NAP (the bias), gross problems in
the surge/residual signal, and whether the model dries a cell the observations
show as wet. Real tidal validation needs sub-hourly output and a harmonic
analysis (e.g. `utide`) of both series.

**Fig. S5**: left, the 10-min record (grey band = the tide), the 00:00
observation (ink) and the model (colour); right, the difference model −
observed at 00:00, where the ~15-day beat is the M2 alias, not model drift.

In [ ]:
if PAIRED.get("elev"):
    display(tables["elev"][["n", "obs_mean", "mod_mean", "bias", "rmse", "r", "std_ratio"]].round(3))
    stns = sorted(PAIRED["elev"], key=lambda k: -len(PAIRED["elev"][k]))
    fig, axes = fs.figure("double", 16 + 24 * len(stns), nrows=len(stns), ncols=2, sharex=True,
                          squeeze=False, width_ratios=[1.6, 1])
    csv = []
    for k, c_ in enumerate(stns):
        df = ts_panel(axes[k, 0], c_, "elev", "wl", PAIRED["elev"])
        diff = (PAIRED["elev"][c_]["mod"] - PAIRED["elev"][c_]["obs"]).reindex(mtimes)
        ax = axes[k, 1]
        ax.axhline(0, color=fs.INK, lw=0.5)
        ax.plot(diff.index, diff.values, color=MODEL_COLOUR, lw=0.8)
        fs.stats_line(ax, f"mean {fs.signed(np.nanmean(diff.values))} m")
        ax.yaxis.set_major_locator(mticker.MaxNLocator(3))
        for a in axes[k]:
            month_axis(a, k == len(stns) - 1)
        axes[k, 0].set_ylabel("Water level (m)")
        ax.set_ylabel("Model − obs. (m)")
        csv.append(df.assign(model_minus_obs=diff.values))
    ts_legend(fig, "elev", "wl")
    for ax, l in zip(axes.flat, "abcdefghijklmnopqrstuvwxyz"):
        fs.panel_label(ax, l)
    fs.save_figure(fig, "figS5_wave_waterlevel", FIG_DIR, data=pd.concat(csv, ignore_index=True))
else:
    print("no water-level pairs")

---
## 11. Currents — ready, but empty for 2015

No RWS current observations exist in the Dutch Wadden Sea in 2015. This cell runs
unchanged for a year ≥ 2020 (Eemshaven from 2020, Den Helder from 2025). Before
trusting the result: GETM's `u`/`v` are documented as velocity in global x/y
direction (already geographic, so no `convc` rotation), and the Eemshaven
bin-averaged `u`/`v` may be a cross-channel mean better compared with a GETM
transect average than with one cell.

In [ ]:
cur_stn = [c_ for c_ in STN if {"u", "v"} <= set(OBS[c_].columns) and OBS[c_]["u"].notna().any()]
if not cur_stn:
    print(f"No current observations in {YEAR} - nothing to compare. Re-run with YEAR >= 2020.")
else:
    rows = []
    for c_ in cur_stn:
        for comp in ("u", "v"):
            o = obs_at(OBS[c_][comp], mtimes, "instantaneous")   # u/v are not averaged
            m = pd.Series(MOD[comp].sel(station=c_).values, index=mtimes)
            d = pd.DataFrame({"obs": o, "mod": m}).dropna()
            if len(d) >= MIN_PAIRS:
                rows.append({"code": c_, "component": comp, **skill(d["obs"], d["mod"])})
    if rows:
        display(pd.DataFrame(rows).set_index(["code", "component"]).round(3))
    print("\nReminder: daily 00:00 samples alias the tidal currents exactly as they alias the water level.")

---
## 12. Wind forcing sanity check

Wave errors usually trace back to the wind. This compares the model's own forcing
(`EUWIND`/`EVWIND`, at the wet cell nearest each KNMI station) with KNMI wind
speed — a check on the **forcing**, not on the model.

In [ ]:
kn_meta = pd.read_csv(OBS_ROOT / "processed" / "knmi_wind_stations.csv")
rows = []
for _, k in kn_meta.iterrows():
    dist, idx = tree.query(_xy(float(k["lon"]), float(k["lat"])))
    if dist > 15:                       # KNMI stations are on land; allow slack
        continue
    j, i = int(jw[idx]), int(iw[idx])
    w = pd.read_csv(OBS_ROOT / "processed" / "knmi_wind" / f"knmi_{int(k['stn'])}.csv.gz",
                    index_col=0, parse_dates=True).loc[f"{YEAR}-01-01":f"{YEAR}-12-31"]
    if w.empty:
        continue
    mu, mv = [], []
    for f in files:
        with xr.open_dataset(f) as d:
            if "EUWIND" not in d.variables:
                break
            mu.append(d["EUWIND"].isel(yc=j, xc=i).to_series())
            mv.append(d["EVWIND"].isel(yc=j, xc=i).to_series())
    if not mu:
        print("model has no EUWIND/EVWIND")
        break
    mu, mv = pd.concat(mu), pd.concat(mv)
    mu, mv = mu[~mu.index.duplicated()], mv[~mv.index.duplicated()]
    how = OBS_CONVENTION.get("Hs_out", "instantaneous")
    spd_o = np.hypot(obs_at(w["wind_u"], mu.index, how), obs_at(w["wind_v"], mv.index, how))
    d = pd.DataFrame({"obs": spd_o, "mod": np.hypot(mu, mv)}).dropna()
    if len(d) >= MIN_PAIRS:
        rows.append({"knmi": f"{int(k['stn'])} {k['label']}", "dist_km": round(float(dist), 1),
                     **skill(d["obs"], d["mod"])})
if rows:
    display(pd.DataFrame(rows).set_index("knmi")[["dist_km", "n", "obs_mean", "mod_mean",
                                                  "bias", "rmse", "r"]].round(3))
    print("\nA systematic wind-speed bias propagates into Hs roughly as Hs ~ U^2 in fetch-limited conditions.")
else:
    print("no KNMI/model wind pairs")

---
## 13. Export

In [ ]:
match.to_csv(OUT_DIR / f"station_matching_{YEAR}.csv")
conv.to_csv(OUT_DIR / f"sampling_convention_{YEAR}.csv")
for mv, t in tables.items():
    t.round(4).to_csv(OUT_DIR / f"skill_{mv}_{YEAR}.csv")
long = pd.DataFrame([{"variable": mv, "observation": PAIRS.get(mv, ""), "station": stn,
                      **row.to_dict()} for mv, t in tables.items() for stn, row in t.iterrows()])
long.round(4).to_csv(OUT_DIR / f"skill_all_{YEAR}.csv", index=False)
print(f"skill_all_{YEAR}.csv: {len(long)} rows ({long['variable'].nunique()} variables)")
try:                                   # openpyxl is often missing on the cluster
    with pd.ExcelWriter(OUT_DIR / f"validation_summary_{YEAR}.xlsx") as xl:
        match.to_excel(xl, sheet_name="station_matching")
        conv.to_excel(xl, sheet_name="sampling_convention")
        for mv, t in tables.items():
            t.round(4).to_excel(xl, sheet_name=mv[:31])
    print("wrote validation_summary.xlsx")
except Exception as exc:
    print(f"skipped the .xlsx ({type(exc).__name__}: {exc}); the CSVs hold the same content")

print("\nfigures in", FIG_DIR)
for f in sorted(FIG_DIR.glob("fig*.pdf")):
    print(f"   {f.stem}")

---
## 14. Reading the results

**Headline numbers.** For each variable: bias, RMSE, scatter index and
correlation from the `** ALL **` row. SI is the most comparable across studies;
for Hs in coastal models, SI below ~0.3 is respectable.

| Symptom | Likely cause |
|---|---|
| `std_ratio` < 1 and `slope` < 1 everywhere | model under-responds — check the wind forcing (§12) or whitecapping |
| Bias grows with Hs (Fig. 6) | growth/dissipation calibration, not a mean offset |
| Bias only at flat stations, fine offshore | depth-limited breaking or bathymetry, not the wave physics |
| Bias range excludes zero (Fig. 7) | model or forcing error, not station matching |
| `elev` bias constant across stations | datum offset between GETM reference level and NAP |

**Before drawing conclusions**, confirm the time zone (RWS observations are UTC;
GETM time is assumed UTC), the sampling convention chosen in §6 (inferred from
the data, not from documentation), and nearest-cell matching at 500 m (`dist_km`
and `model_depth_m` in §4 are the first place to look for an outlier station).

**Cheapest extensions**: hourly model output (unlocks tidal validation and
removes the aliasing caveat), more years (storm statistics), and a QQ plot of
the extremes, where a wave model usually fails first.